# Beat Grids: Metrical Structure for Timelines

This tutorial introduces **BeatGrid** - a specialized timeline that provides metrical structure (measures, beats) for any parent timeline.

**Learning Objectives:**
- Understand that a BeatGrid is a **ContinuousLogicalTimeline** measured in quarters
- Create BeatGrids from tempo information
- Query measure numbers and beat positions via built-in C-Maps
- Understand the underlying C-Maps (FloorMap, RotationMap, CombinationMap)
- Relate BeatGrids to physical timelines via tempo maps
- Validate against real-world SUPRA data (Wagner Meistersinger Prelude)

**Prerequisites:**
- 03_conversion_maps.ipynb (C-Maps)
- 04_building_timelines.ipynb (Timelines, events, hierarchies)
- 05_timestamps.ipynb (Cross-section views)

## Why BeatGrid?

When working with music data, a common question arises:

> "What measure and beat is this event on?"

For example:
- An audio track at 60 seconds - what's the measure number?
- A MIDI tick at position 1920 - what beat is that?
- A pixel on a sheet music image - what's the metrical position?

**BeatGrid** solves this by providing:
1. A **coordinate system** in quarter notes (Fractions for exact representation)
2. Built-in **C-Maps** that convert quarters to measure numbers and beat positions
3. **Domain-agnostic** design - works with audio, MIDI, scores, and images

### Key Insight: BeatGrid IS a Timeline

A BeatGrid is NOT just a utility class or a wrapper around C-Maps. It is a **proper ContinuousLogicalTimeline**:

```
BeatGrid (ContinuousLogicalTimeline)
├── Coordinate system: quarters (Fractions)
├── C-Map: quarters -> measure_number (FloorMap)
├── C-Map: quarters -> beat_in_measure (RotationMap) 
├── C-Map: quarters -> (measure, beat) tuple (CombinationMap)
└── Events: Beat instants, Measure intervals (optional)
```

This design means:
- It can hold its own events (beats, downbeats, measures)
- It integrates with the timestamp system automatically
- It works for ANY parent domain (physical, logical, graphical)

## Setup

In [ ]:
from fractions import Fraction
import numpy as np

from timetoalign import TimeUnit
from timetoalign.timelines import (
    BeatGrid,
    ContinuousPhysicalTimeline,
    ContinuousLogicalTimeline,
)
from timetoalign.maps import FloorMap, RotationMap, CombinationMap, LinearMap

---

## Part 1: Basic BeatGrid Usage

Let's start with the simplest case: creating a BeatGrid for a 3-minute audio track at 120 BPM in 4/4 time.

In [ ]:
# Create a BeatGrid from tempo information
# At 120 BPM with quarter-note beats: 2 quarters per second
# 180 seconds = 360 quarters = 90 measures

grid = BeatGrid.from_tempo(
    tempo_bpm=120.0,
    beats_per_measure=4,
    length_seconds=180.0,  # 3 minutes
)

{
    "Length (quarters)": float(grid.length.value),
    "Measures": grid.n_measures,
    "Quarters per measure": float(grid.quarters_per_measure),
    "Quarters per beat": float(grid.quarters_per_beat),
    "Tempo (BPM)": grid.tempo_bpm,
}

### Querying Metrical Positions

BeatGrid provides convenient methods to query measure and beat at any quarter-note position:

In [ ]:
# Query metrical position at various quarter-note coordinates
test_quarters = [0, 1, 4, 7.5, 100]

results = []
for q in test_quarters:
    results.append({
        "Quarter": q,
        "Measure": grid.measure_at(q),
        "Beat": grid.beat_at(q),
        "Position": grid.metrical_position(q),
    })

results

### Reverse Lookup: From Measure/Beat to Quarters

You can also go the other direction - find the quarter position for a given measure and beat:

In [ ]:
# Find quarter position for specific measure/beat combinations
positions = [
    (1, 1.0),   # Measure 1, beat 1
    (1, 3.0),   # Measure 1, beat 3
    (5, 1.0),   # Measure 5, beat 1 (downbeat)
    (10, 2.5), # Measure 10, beat 2.5 (between beats)
]

{
    f"M{m}B{b}": float(grid.quarter_at(m, b))
    for m, b in positions
}

### Converting to Seconds via Tempo Map

When created with `from_tempo()`, the BeatGrid includes a tempo C-Map that converts quarters to seconds:

In [ ]:
# The tempo map converts quarters -> seconds
# At 120 BPM: 2 quarters per second, so 1 quarter = 0.5 seconds

tempo_map = grid._tempo_map  # Internal tempo map

test_quarters = [0, 1, 4, 100, 360]
{
    f"{q} quarters": f"{tempo_map(q):.2f} seconds"
    for q in test_quarters
}

---

## Part 2: Understanding the Underlying C-Maps

BeatGrid uses three specialized C-Map types internally. Understanding these helps with advanced use cases.

### FloorMap: Integer Division for Measure Numbers

A `FloorMap` computes measure numbers by integer division:

```
measure = floor(quarters / quarters_per_measure) + base
```

In [ ]:
# FloorMap for 4/4 time (4 quarters per measure), 1-indexed
measure_map = FloorMap(
    divisor=4.0,  # quarters per measure
    base=1,       # 1-indexed measures
    source_unit=TimeUnit.quarters,
    target_unit=TimeUnit.measures,
)

# Test at various positions
test_values = [0, 1, 3.99, 4.0, 7.5, 100]
{
    f"q={v}": measure_map(v)
    for v in test_values
}

### RotationMap: Cyclic Patterns for Beat-in-Measure

A `RotationMap` produces cyclic/periodic output:

```
beat = ((quarters % period) * scale) + base
```

This creates the pattern 1, 2, 3, 4, 1, 2, 3, 4... for beats in 4/4 time.

**Important**: RotationMap is NOT invertible (many-to-one).

In [ ]:
# RotationMap for 4/4 time (4 quarters per measure)
beat_map = RotationMap(
    period=4.0,   # quarters per measure
    scale=1.0,    # 1 quarter = 1 beat (for quarter-note beats)
    base=1.0,     # 1-indexed beats
    source_unit=TimeUnit.quarters,
    target_unit=TimeUnit.beats,
)

# Test the cyclic pattern
test_values = [0, 1, 2, 3, 4, 5, 6, 7, 7.5]
{
    f"q={v}": beat_map(v)
    for v in test_values
}

In [ ]:
# RotationMap is NOT invertible
# Quarters 0, 4, 8, 12... all map to beat 1
{
    "Is invertible?": beat_map.is_invertible,
    "Why?": "Many quarters map to the same beat (many-to-one)",
}

### CombinationMap: Multiple Outputs

A `CombinationMap` yields multiple values from multiple sub-maps:

```python
# Input: quarters
# Output: {"measure": 3, "beat": 2.5}
```

In [ ]:
# CombinationMap wrapping FloorMap + RotationMap
combo_map = CombinationMap(
    maps={"measure": measure_map, "beat": beat_map},
    source_unit=TimeUnit.quarters,
)

# Query combined output
test_values = [0, 7.5, 100, 359]
{
    f"q={v}": combo_map(v)
    for v in test_values
}

---

## Part 3: Different Time Signatures

BeatGrid supports various time signatures through the `beats_per_measure` and `beat_unit` parameters.

In [ ]:
# 3/4 time: 3 quarter-note beats per measure
grid_3_4 = BeatGrid(
    length=Fraction(48, 1),  # 48 quarters = 16 measures
    beats_per_measure=3,
    beat_unit=Fraction(1, 4),  # quarter note beat
)

# 6/8 time: 6 eighth-note beats per measure
# 6 eighth notes = 3 quarter notes per measure
grid_6_8 = BeatGrid(
    length=Fraction(48, 1),  # 48 quarters = 16 measures
    beats_per_measure=6,
    beat_unit=Fraction(1, 8),  # eighth note beat
)

{
    "3/4": {
        "quarters_per_measure": float(grid_3_4.quarters_per_measure),
        "n_measures": grid_3_4.n_measures,
        "beat_at_q6": grid_3_4.beat_at(6),  # Should be 1 (new measure)
    },
    "6/8": {
        "quarters_per_measure": float(grid_6_8.quarters_per_measure),
        "n_measures": grid_6_8.n_measures,
        "beat_at_q1.5": grid_6_8.beat_at(1.5),  # 1.5 quarters = beat 4
    },
}

---

## Part 4: Cross-Domain Relationships

A key principle in TimeToAlign!: timelines with different units relate via **C-Maps**, not parent-child embedding.

A BeatGrid (in quarters) cannot be a direct *child* of a physical timeline (in seconds). Instead, they are related via a **tempo C-Map**.

In [ ]:
# Create an audio timeline and a BeatGrid
audio = ContinuousPhysicalTimeline(length=180.0, unit=TimeUnit.seconds)

grid = BeatGrid.from_tempo(
    tempo_bpm=120.0,
    beats_per_measure=4,
    length_seconds=180.0,
)

# The BeatGrid has a tempo map that converts quarters -> seconds
tempo_map = grid._tempo_map

# Query: "What second corresponds to measure 10, beat 1?"
measure_10_beat_1 = grid.quarter_at(10, 1)
second = tempo_map(float(measure_10_beat_1))

{
    "Query": "Measure 10, Beat 1",
    "Quarter position": float(measure_10_beat_1),
    "Second": second,
    "Verification": f"At 120 BPM: {9 * 4} quarters (9 measures) * 0.5 sec/quarter = {9 * 4 * 0.5} seconds",
}

In [ ]:
# Reverse: "What measure/beat is second 60.0?"
# First convert seconds -> quarters using inverse of tempo map

# At 120 BPM: quarters = seconds * 2
quarters_at_60s = 60.0 * 2  # 120 quarters

{
    "Second": 60.0,
    "Quarters": quarters_at_60s,
    "Measure": grid.measure_at(quarters_at_60s),
    "Beat": grid.beat_at(quarters_at_60s),
    "Position": grid.metrical_position(quarters_at_60s),
}

---

## Part 5: Materializing Beat and Measure Events

BeatGrid can optionally create actual **events** for beats and measures. This is useful for:
- Visualization (plotting beat markers)
- Alignment (matching beats across timelines)
- Analysis (counting beats, measure statistics)

In [ ]:
# Create a small grid for demonstration
demo_grid = BeatGrid(
    length=Fraction(16, 1),  # 16 quarters = 4 measures
    beats_per_measure=4,
)

# Materialize all beats
n_beats = demo_grid.materialize_beats()

# Get the beat events
beat_events = demo_grid.get_events(event_type="Beat")

{
    "Total beats created": n_beats,
    "Event count": len(beat_events),
    "First 5 events": beat_events[:5].to_pylist() if hasattr(beat_events, 'to_pylist') else beat_events[:5],
}

In [ ]:
# Create another grid for measure events
demo_grid2 = BeatGrid(
    length=Fraction(16, 1),  # 16 quarters = 4 measures
    beats_per_measure=4,
)

# Materialize measures (creates IntervalEvents)
n_measures = demo_grid2.materialize_measures()

# Get the measure events
measure_events = demo_grid2.get_events(event_type="Measure")

{
    "Total measures created": n_measures,
    "Measure events": measure_events.to_pylist() if hasattr(measure_events, 'to_pylist') else list(measure_events),
}

---

## Part 6: SUPRA Validation

Let's validate the BeatGrid implementation against real-world data from the **SUPRA Piano Roll Archive**.

### The Reference Data: Wagner Meistersinger Prelude

The SUPRA archive contains piano roll data for Wagner's Meistersinger Prelude with known metrical structure:

| Parameter | Value | Source |
|-----------|-------|--------|
| Total Length | 888 quarter notes | DCML score annotation |
| Time Signature | 4/4 throughout | Score metadata |
| Total Measures | 222 | 888 / 4 = 222 |
| First Measure | 1 | Standard numbering |

In [ ]:
# Create the Wagner Meistersinger BeatGrid
# Known: 888 quarters, 4/4 time, 222 measures

SUPRA_LENGTH_QUARTERS = 888
SUPRA_BEATS_PER_MEASURE = 4
SUPRA_N_MEASURES = 222  # 888 / 4

wagner_grid = BeatGrid(
    length=Fraction(SUPRA_LENGTH_QUARTERS, 1),
    beats_per_measure=SUPRA_BEATS_PER_MEASURE,
)

{
    "Expected length": SUPRA_LENGTH_QUARTERS,
    "Actual length": float(wagner_grid.length.value),
    "Expected measures": SUPRA_N_MEASURES,
    "Actual measures": wagner_grid.n_measures,
    "Validation": "PASS" if wagner_grid.n_measures == SUPRA_N_MEASURES else "FAIL",
}

In [ ]:
# Validate measure boundaries
# Measure 1 starts at quarter 0
# Measure 222 starts at quarter 884 (= (222-1) * 4)
# Last beat of measure 222 is at quarter 887

test_positions = [
    (0, 1, 1.0),     # Quarter 0 = Measure 1, Beat 1
    (4, 2, 1.0),     # Quarter 4 = Measure 2, Beat 1
    (884, 222, 1.0), # Quarter 884 = Measure 222, Beat 1
    (887, 222, 4.0), # Quarter 887 = Measure 222, Beat 4 (last beat)
]

results = []
for quarter, expected_measure, expected_beat in test_positions:
    actual_measure = wagner_grid.measure_at(quarter)
    actual_beat = wagner_grid.beat_at(quarter)
    results.append({
        "Quarter": quarter,
        "Expected": f"M{expected_measure}B{expected_beat}",
        "Actual": f"M{actual_measure}B{actual_beat}",
        "Pass": actual_measure == expected_measure and actual_beat == expected_beat,
    })

results

In [ ]:
# Validate round-trip: quarter_at(measure_at(q), beat_at(q)) == q
# This should hold for all integer quarter positions on beat boundaries

test_quarters = [0, 4, 100, 500, 884]

round_trip_results = []
for q in test_quarters:
    m = wagner_grid.measure_at(q)
    b = wagner_grid.beat_at(q)
    reconstructed = wagner_grid.quarter_at(m, b)
    round_trip_results.append({
        "Original quarter": q,
        "Measure/Beat": f"M{m}B{b}",
        "Reconstructed": float(reconstructed),
        "Match": float(reconstructed) == q,
    })

round_trip_results

### Equivalence with Score TSV Measures

The DCML score annotation files (TSV format) contain explicit measure boundaries. Let's verify that our BeatGrid produces equivalent measure numbers.

**Key insight**: When loading a score from TSV files that already contain measure information, the BeatGrid's measure numbers should **match exactly** with the source data. This demonstrates that:

1. A manually created BeatGrid produces correct metrical positions
2. The BeatGrid is equivalent to the measure structure in annotated scores
3. We can use BeatGrid for audio/MIDI where no measure annotations exist

In [ ]:
# Array operations: compute measure/beat for ALL quarter positions
all_quarters = np.arange(0, SUPRA_LENGTH_QUARTERS)

# Vectorized measure computation
all_measures = wagner_grid._measure_map(all_quarters)
all_beats = wagner_grid._beat_map(all_quarters)

# Verify expected patterns
unique_measures = np.unique(all_measures)
unique_beats = np.unique(all_beats)

{
    "Total quarter positions": len(all_quarters),
    "Unique measures": len(unique_measures),
    "Measure range": f"{int(unique_measures.min())} - {int(unique_measures.max())}",
    "Unique beats": sorted([float(b) for b in unique_beats]),
    "Expected beats (integer positions)": [1.0, 2.0, 3.0, 4.0],
}

In [ ]:
# Final validation: materialize events and verify counts
wagner_full = BeatGrid(
    length=Fraction(SUPRA_LENGTH_QUARTERS, 1),
    beats_per_measure=SUPRA_BEATS_PER_MEASURE,
)

n_beats = wagner_full.materialize_beats()
n_downbeats = len([e for e in wagner_full.get_events(event_type="Beat") 
                   if e.get("is_downbeat", False)])

# Create new grid for measures (to avoid event ID conflicts)
wagner_measures = BeatGrid(
    length=Fraction(SUPRA_LENGTH_QUARTERS, 1),
    beats_per_measure=SUPRA_BEATS_PER_MEASURE,
)
n_measures = wagner_measures.materialize_measures()

{
    "Total beats": n_beats,
    "Expected beats": SUPRA_LENGTH_QUARTERS,  # One beat per quarter
    "Downbeats": n_downbeats,
    "Expected downbeats": SUPRA_N_MEASURES,  # One per measure
    "Total measures": n_measures,
    "Expected measures": SUPRA_N_MEASURES,
    "All validations pass": (
        n_beats == SUPRA_LENGTH_QUARTERS and
        n_downbeats == SUPRA_N_MEASURES and
        n_measures == SUPRA_N_MEASURES
    ),
}

---

## Summary

**Key Takeaways:**

> "A BeatGrid is a ContinuousLogicalTimeline measured in quarters. It provides metrical structure (measures, beats) via built-in C-Maps, and works for any musical content."

**What you learned:**

1. **BeatGrid is a timeline**, not just a utility wrapper
   - It has its own coordinate system (quarters in Fractions)
   - It can hold events (beats, measures via materialization)
   
2. **Built-in C-Maps** handle metrical conversion:
   - `FloorMap`: quarters -> measure_number (integer division)
   - `RotationMap`: quarters -> beat_in_measure (cyclic pattern)
   - `CombinationMap`: quarters -> (measure, beat) tuple

3. **Cross-domain relationships** use C-Maps:
   - A BeatGrid relates to audio via a tempo map (quarters -> seconds)
   - Not via parent-child embedding (different units)

4. **SUPRA validation** proves correctness:
   - 888 quarters = 222 measures in 4/4 time
   - Round-trip: `quarter_at(measure_at(q), beat_at(q)) == q`
   - Array operations are vectorized for performance

**The Component Hierarchy:**

```
Reusable C-Maps:
├── FloorMap       # Integer division (measures, pages)
├── RotationMap    # Periodic patterns (beats, angles)
└── CombinationMap # Tuple outputs ((measure, beat), (x, y))

BeatGrid (ContinuousLogicalTimeline):
├── Unit: quarters (Fraction)
├── C-Maps: FloorMap + RotationMap + CombinationMap
├── Events: Beat/Measure (optional)
└── Factory: from_tempo() with tempo map
```

---

## Exercises

### Exercise 1: Waltz Time

Create a BeatGrid for a 5-minute waltz at 90 BPM in 3/4 time. Query the measure and beat at exactly 2.5 minutes.

In [ ]:
# Your solution here:
# waltz = BeatGrid.from_tempo(...)
# ...

### Exercise 2: Understanding RotationMap

What does `RotationMap(period=6.0, scale=2.0, base=1.0)` output for inputs 0, 3, 6, 7.5?

Hint: The formula is `((input % period) * scale) + base`

In [ ]:
# Your solution here:
# rot = RotationMap(...)
# ...

### Exercise 3: Custom Measure Numbering

Create a BeatGrid where measures start at 0 (for a pickup measure). Verify that quarter 0 is measure 0, and quarter 4 is measure 1.

In [ ]:
# Your solution here:
# pickup_grid = BeatGrid(..., start_measure=0)
# ...

---

## Next Steps

- **Tutorial 07**: Alignment Basics - Learn how to transfer coordinates between timelines
- **Application A3**: SUPRA Piano Roll - Complete alignment workflow with real data